# 04 - Rules, models and selection on the validation period
* **Business rules (non-ML):** shortest promised window, longest distance, highest freight ratio, worst seller history.
* **Learned models:** logistic regression (simple, interpretable baseline) and LightGBM (gradient-boosted trees).
* **Selection criterion (validation, Mar-May 2018):** number of late orders captured by the daily top-10 policy
  (= daily precision@10); ties broken by PR-AUC. The test period is not touched in this notebook.

In [1]:
# Cell 1 - make the project package importable and define folders
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Image
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 80)
from olist_risk import config
config.ensure_dirs()
print("PROJECT_ROOT =", PROJECT_ROOT)
print("outputs ->", config.OUTPUT_DIR)

PROJECT_ROOT = /workspace/ml_project2_v2
outputs -> /workspace/ml_project2_v2/outputs


In [2]:
# Cell 2 - load features, define training and validation sets
import json, time, joblib
from olist_risk import models, policy, metrics, features
ft = pd.read_pickle(config.PROCESSED_DIR / "features.pkl")
train = ft[ft.train_eligible].copy(); y_train = train.is_late.astype(int)
val_pool = ft[ft.split == "validation"].copy()            # daily candidate pool (all approved orders)
val_lab = val_pool[val_pool.label_available]               # labelled subset for predictive metrics
print(f"train rows {len(train):,} (late rate {y_train.mean():.4f}) approved {train.order_approved_at.min().date()} .. {train.order_approved_at.max().date()}")
print(f"validation pool {len(val_pool):,} orders on {val_pool.approval_date.nunique()} days; labelled {len(val_lab):,} (late rate {val_lab.is_late.mean():.4f})")

train rows 43,276 (late rate 0.0562) approved 2017-01-05 .. 2017-12-30
validation pool 21,132 orders on 89 days; labelled 20,662 (late rate 0.1026)


In [3]:
# Cell 3 - business rules and a random reference on validation (same daily top-10 policy)
rows = []
for r, f in policy.RULES.items():
    val_pool[r] = f(val_pool).values
    e = policy.evaluate_policy(val_pool, r)
    pm = metrics.predictive_metrics(val_lab.is_late, f(val_lab), probabilistic=False)
    rows.append({"method": r, "family": "rule", **{k: e[k] for k in ["late_identified", "unnecessary_investigations", "undetermined_selected", "late_missed", "precision"]},
                 "pr_auc": pm["pr_auc"], "roc_auc": pm["roc_auc"]})
rnd = policy.random_policy_expectation(val_pool, n_rep=200)
rows.append({"method": "random (mean of 200 draws)", "family": "reference", "late_identified": rnd["late_identified"],
             "unnecessary_investigations": rnd["unnecessary_investigations"], "undetermined_selected": rnd["undetermined_selected"],
             "late_missed": rnd["late_missed"], "precision": rnd["precision"], "pr_auc": val_lab.is_late.mean(), "roc_auc": 0.5})
rules_val = pd.DataFrame(rows); display(rules_val.round(4))
best_rule = rules_val[rules_val.family == "rule"].sort_values(["late_identified", "pr_auc"], ascending=False).iloc[0].method
print("best rule on validation ->", best_rule)

,method,family,late_identified,unnecessary_investigations,undetermined_selected,late_missed,precision,pr_auc,roc_auc
0,rule_shortest_promise,rule,46.00,826.000,18.000,2074.00,0.0517,0.0985,0.5093
1,rule_longest_distance,rule,202.00,655.000,33.000,1918.00,0.2270,0.1675,0.6455
2,rule_highest_freight_ratio,rule,119.00,751.000,20.000,2001.00,0.1337,0.1207,0.5485
3,rule_worst_seller_history,rule,123.00,737.000,30.000,1997.00,0.1382,0.0969,0.4761
4,random (mean of 200 draws),reference,87.79,782.325,19.885,2032.21,0.0986,0.1026,0.5000


best rule on validation -> rule_longest_distance


In [4]:
# Cell 4 - hyper-parameter and feature-set search for both learned models (fit on train, score validation)
grid_rows = []
for fs_name, fs in models.FEATURE_SETS.items():
    for kind, grid in [("logreg", models.LOGREG_GRID), ("lgbm", models.LGBM_GRID)]:
        for params in grid:
            t0 = time.time()
            m = models.FittedModel(kind, params, num_features=fs).fit(train, y_train)
            val_pool["_s"] = m.score(val_pool)
            e = policy.evaluate_policy(val_pool, "_s")
            pm = metrics.predictive_metrics(val_lab.is_late, m.score(val_lab))
            grid_rows.append({"model": kind, "feature_set": fs_name, "params": json.dumps(params), "late_identified": e["late_identified"],
                              "unnecessary_investigations": e["unnecessary_investigations"], "precision_at_10_daily": e["precision"],
                              "pr_auc": pm["pr_auc"], "roc_auc": pm["roc_auc"], "brier": pm["brier"], "fit_seconds": round(time.time() - t0, 1)})
grid = pd.DataFrame(grid_rows)
grid.to_csv(config.TABLE_DIR / "04_validation_grid.csv", index=False)
print(len(grid), "configurations evaluated")
display(grid.sort_values(["late_identified", "pr_auc"], ascending=False).head(10).round(4))

36 configurations evaluated


,model,feature_set,params,late_identified,unnecessary_investigations,precision_at_10_daily,pr_auc,roc_auc,brier,fit_seconds
18,logreg,no_market,"{""C"": 0.01, ""class_weight"": null}",255,602,0.2865,0.2011,0.6974,0.0885,2.7
0,logreg,all,"{""C"": 0.01, ""class_weight"": null}",247,605,0.2775,0.2234,0.7246,0.0867,2.6
20,logreg,no_market,"{""C"": 0.1, ""class_weight"": null}",247,605,0.2775,0.2079,0.7011,0.0883,6.8
19,logreg,no_market,"{""C"": 0.01, ""class_weight"": ""balanced""}",243,611,0.2730,0.2063,0.7023,0.2698,4.5
22,logreg,no_market,"{""C"": 1.0, ""class_weight"": null}",239,615,0.2685,0.2058,0.6981,0.0888,17.4
21,logreg,no_market,"{""C"": 0.1, ""class_weight"": ""balanced""}",232,622,0.2607,0.2029,0.6988,0.2469,10.7
2,logreg,all,"{""C"": 0.1, ""class_weight"": null}",230,621,0.2584,0.2024,0.6923,0.0885,6.9
7,lgbm,all,"{""num_leaves"": 7, ""n_estimators"": 200, ""min_child_samples"": 400, ""learning_r...",227,632,0.2551,0.1958,0.6631,0.0886,0.4
23,logreg,no_market,"{""C"": 1.0, ""class_weight"": ""balanced""}",226,629,0.2539,0.2006,0.6961,0.2413,22.4
1,logreg,all,"{""C"": 0.01, ""class_weight"": ""balanced""}",226,624,0.2539,0.1798,0.6606,0.2940,4.5


In [5]:
# Cell 5 - best configuration per model family (selected on validation only)
best = (grid.sort_values(["late_identified", "pr_auc"], ascending=False).groupby("model").head(1).set_index("model"))
display(best.round(4))
selected_model = best.sort_values(["late_identified", "pr_auc"], ascending=False).index[0]
print("proposed model (best on validation):", selected_model)

,feature_set,params,late_identified,unnecessary_investigations,precision_at_10_daily,pr_auc,roc_auc,brier,fit_seconds
model,,,,,,,,,
logreg,no_market,"{""C"": 0.01, ""class_weight"": null}",255,602,0.2865,0.2011,0.6974,0.0885,2.7
lgbm,all,"{""num_leaves"": 7, ""n_estimators"": 200, ""min_child_samples"": 400, ""learning_r...",227,632,0.2551,0.1958,0.6631,0.0886,0.4


proposed model (best on validation): logreg


In [6]:
# Cell 6 - validation comparison table: rules vs learned models under the same daily cap
val_cmp = pd.concat([rules_val, pd.DataFrame([{"method": f"{k} (best config)", "family": "learned", "late_identified": r.late_identified,
          "unnecessary_investigations": r.unnecessary_investigations, "precision": r.precision_at_10_daily, "pr_auc": r.pr_auc,
          "roc_auc": r.roc_auc} for k, r in best.iterrows()])], ignore_index=True)
val_cmp.to_csv(config.TABLE_DIR / "04_validation_comparison.csv", index=False)
display(val_cmp.round(4))

,method,family,late_identified,unnecessary_investigations,undetermined_selected,late_missed,precision,pr_auc,roc_auc
0,rule_shortest_promise,rule,46.00,826.000,18.000,2074.00,0.0517,0.0985,0.5093
1,rule_longest_distance,rule,202.00,655.000,33.000,1918.00,0.2270,0.1675,0.6455
2,rule_highest_freight_ratio,rule,119.00,751.000,20.000,2001.00,0.1337,0.1207,0.5485
3,rule_worst_seller_history,rule,123.00,737.000,30.000,1997.00,0.1382,0.0969,0.4761
4,random (mean of 200 draws),reference,87.79,782.325,19.885,2032.21,0.0986,0.1026,0.5000
5,logreg (best config),learned,255.00,602.000,NaN,NaN,0.2865,0.2011,0.6974
6,lgbm (best config),learned,227.00,632.000,NaN,NaN,0.2551,0.1958,0.6631


In [7]:
# Cell 7 - final refit of the selected configurations on train+validation (labels mature before TEST_START)
trainval = ft[ft.trainval_eligible].copy(); y_tv = trainval.is_late.astype(int)
print(f"refit rows {len(trainval):,} (late rate {y_tv.mean():.4f}), approved up to {trainval.order_approved_at.max().date()}")
final = {}
for kind, r in best.iterrows():
    fs = models.FEATURE_SETS[r.feature_set]
    final[kind] = models.FittedModel(kind, json.loads(r.params), num_features=fs).fit(trainval, y_tv)
    joblib.dump(final[kind], config.MODEL_DIR / f"{kind}_final.joblib")
    # validation-period model kept for the 'refit vs no refit' diagnostic in notebook 06
    joblib.dump(models.FittedModel(kind, json.loads(r.params), num_features=fs).fit(train, y_train), config.MODEL_DIR / f"{kind}_trainonly.joblib")
print("saved:", sorted(p.name for p in config.MODEL_DIR.glob("*.joblib")))

refit rows 64,109 (late rate 0.0796), approved up to 2018-04-01


saved: ['lgbm_final.joblib', 'lgbm_trainonly.joblib', 'logreg_final.joblib', 'logreg_trainonly.joblib']


In [8]:
# Cell 8 - experiment configuration record (traceability)
exp = {
    "seed": config.SEED, "splits": {"train": [config.TRAIN_START, config.VAL_START], "validation": [config.VAL_START, config.TEST_START],
                                    "test": [config.TEST_START, config.TEST_END]},
    "label_definition": "is_late = date(order_delivered_customer_date) > date(order_estimated_delivery_date); undelivered = undetermined",
    "label_maturity_days": config.LABEL_MATURITY_DAYS, "daily_capacity": config.DAILY_CAPACITY,
    "train_rows_for_selection": int(len(train)), "trainval_rows_for_final_refit": int(len(trainval)),
    "selection_criterion": "validation late orders captured by daily top-10, tie-break PR-AUC",
    "logreg_grid": models.LOGREG_GRID, "lgbm_grid": models.LGBM_GRID, "feature_sets": {k: v for k, v in models.FEATURE_SETS.items()},
    "categorical_features": features.CAT_FEATURES,
    "selected": {k: {"params": json.loads(r.params), "feature_set": r.feature_set, "val_late_identified": int(r.late_identified),
                     "val_pr_auc": round(r.pr_auc, 4)} for k, r in best.iterrows()},
    "proposed_model": selected_model, "reference_rule": best_rule,
    "libraries": {m.__name__: m.__version__ for m in [np, pd, __import__("sklearn"), __import__("lightgbm")]},
}
json.dump(exp, open(config.OUTPUT_DIR / "experiment_configs.json", "w"), indent=2, default=str)
print(json.dumps({k: exp[k] for k in ["selected", "proposed_model", "reference_rule"]}, indent=2))

{
  "selected": {
    "logreg": {
      "params": {
        "C": 0.01,
        "class_weight": null
      },
      "feature_set": "no_market",
      "val_late_identified": 255,
      "val_pr_auc": 0.2011
    },
    "lgbm": {
      "params": {
        "num_leaves": 7,
        "n_estimators": 200,
        "min_child_samples": 400,
        "learning_rate": 0.03
      },
      "feature_set": "all",
      "val_late_identified": 227,
      "val_pr_auc": 0.1958
    }
  },
  "proposed_model": "logreg",
  "reference_rule": "rule_longest_distance"
}
